# 00 — How good are the rules, actually?

Phase 0's deliverable is **a number**, not a model: how well do PayGuard's six
rules perform against fraud they were not designed to catch?

Two ways to produce it, in order of preference:

1. **Real data.** Replay a labelled dataset through the running stack first:
   ```bash
   ./.venv/bin/python run_baseline.py --replay-csv data/sparkov.csv \
       --kind sparkov --limit 50000 --api-key "$PAYGUID_API_KEY"
   ```
   then point `EXPORT_CSV` at `ml/artifacts/...` or at a
   `GET /api/transactions/export` dump.
2. **Synthetic fixture.** Leave `EXPORT_CSV = None` below. Every number is
   then labelled `synthetic-demo` and is a smoke test, **not a result**.

The discipline this notebook enforces: labels come from disputes / the
dataset, never from `rulesTriggered`; PR-AUC is the headline; accuracy is
printed only to be dismissed.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
from payguard_ml import baseline, dataset, metrics

# Point this at a PayGuard export to produce the REAL baseline.
EXPORT_CSV = None  # e.g. 'data/export.csv'
N_SYNTHETIC = 20_000

In [ ]:
if EXPORT_CSV:
    result = baseline.baseline_from_export(EXPORT_CSV)
    frame = baseline.parse_export_csv(EXPORT_CSV)
else:
    frame = baseline.simulate_rule_engine(dataset.load_synthetic(n=N_SYNTHETIC, seed=7))
    result = baseline.baseline_from_frame(frame, source='synthetic-demo')

print(baseline.render_markdown(result))
if result.source == 'synthetic-demo':
    print('\n>>> FIXTURE: these numbers are a smoke test, not a result.')

In [ ]:
y = frame['is_fraud'].astype(bool).to_numpy()
score = frame['fraudScore'].fillna(0).to_numpy(dtype=float)

p, r, _ = metrics.precision_recall_curve(y, score)
ap = metrics.average_precision(y, score)
prevalence = y.mean()

plt.figure(figsize=(6, 5))
plt.plot(r, p, label=f'rules (PR-AUC={ap:.3f})')
plt.axhline(prevalence, ls='--', c='grey', label=f'random ({prevalence:.3%})')
plt.xlabel('recall'); plt.ylabel('precision')
plt.title('Precision–recall — the rules as a ranker')
plt.legend(); plt.tight_layout(); plt.show()

# The point: a random ranker gets AP == prevalence, so PR-AUC is directly
print(f'prevalence = {prevalence:.4%}  ->  a random model scores AP = {prevalence:.4f}')

In [ ]:
# Reliability: is the engine's severity a probability? (Spoiler: no.)
prob = (score / 100.0)
pred, obs, counts = metrics.calibration_curve(y, prob, n_bins=10)

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], ls='--', c='grey', label='perfect')
plt.plot(pred, obs, 'o-', label='rule severity')
plt.xlabel('predicted probability (severity/100)'); plt.ylabel('observed fraud rate')
plt.title(f"Reliability diagram — ECE={metrics.expected_calibration_error(y, prob):.3f}")
plt.legend(); plt.tight_layout(); plt.show()

## What to take from this

- **High recall is easy; precision is the hard part.** If the rules flag a
  quarter of all traffic to catch most frauds, the analyst queue is the
  bottleneck, and that is what PR-AUC measures and ROC-AUC hides.
- **Some rules are near-worthless.** Rerun with the real data and look at
  `NIGHT_OWL_RULE`'s precision. The learned weights in Phase 1 will put a
  number on exactly this.
- **The severity scale is not a probability.** The reliability diagram shows
  it; Phase 1 fixes it with Platt/isotonic calibration.
- **Do not tune the rules to improve this number.** The gap *is* the project.

Record the outcome in `EXPERIMENTS.md`, then move to Phase 1.